In [15]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import numpy as np
import math
from matplotlib.scale import FuncScale

from pathlib import Path

from os.path import join
import os
from functools import partial
import pathlib
import shutil

import re

import multiprocessing as mp


In [16]:
num_process = 2
verif_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/mpas/mpas_forecasts"
scrubbed_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/mpas/mpas_fixed"

In [17]:
verif_dir = Path(verif_dir)
scrubbed_dir = Path(scrubbed_dir)

verif_dirs = [p for p in verif_dir.iterdir() if p.is_dir() and p.name != "gifs"]

In [18]:
data_stats = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/data_stats.nc")
data_stats["channel"] = [4, 7,8,9,10, 13]
channels = data_stats.channel[1:]
data_stats = data_stats.sel(channel=channels)

# clip and write files

In [19]:
# def to_bool(da):
#     return bool(da.to_array().values)
    

# def process_dir(folder):
#     os.makedirs(join(scrubbed_dir, folder.name), exist_ok=True)
#     for f in folder.iterdir():
#         ds = xr.open_dataset(f)
#         # clip ds
#         ds = ds.sel(channel=channels).clip(min=data_stats["min"], max=data_stats["max"]).load()
#         ds.to_netcdf(join(scrubbed_dir, folder.name, f.name))

# os.makedirs(scrubbed_dir, exist_ok=True)

# with mp.Pool(num_process) as p:
#     p.map(process_dir, verif_dirs)

In [20]:
data_stats["min"].values

array([120.34575653,  78.40976715,  71.56335449,  68.20957947,
        50.20142746])

# check files

In [21]:
def to_bool(da):
    return bool(da.to_array().values)

def check_dir(folder):
    for f in folder.iterdir():
        ds = xr.open_dataset(f)
        # clip ds
        ds = ds.sel(channel=channels)
        ds_max = ds.max(dim=["latitude", "longitude", "t"])
        ds_min = ds.min(dim=["latitude", "longitude", "t"])
        if float(ds_max.max().to_array().values[0]) > 450. or float(ds_min.min().to_array().values[0]) < 50.:
            print(f"error in {f}")
        if to_bool((ds_max > data_stats["max"]).any()) or to_bool((ds_min < data_stats["min"]).any()):
            print(f"error in {f}")

scrubbed_dirs = [p for p in scrubbed_dir.iterdir() if p.is_dir() and p.name != "gifs"]

with mp.Pool(num_process) as p:
    p.map(check_dir, scrubbed_dirs)

KeyboardInterrupt: 

In [25]:
data_stats["min"].sel(channel=13)

<xarray.DataArray 'min' ()>
array(50.201427)
Coordinates:
    channel  int64 13